# Pemodelan Mitigasi Risiko Gagal Panen Berbasis AHP dan Kerangka CROPDAM-X
**Penelitian Magister Teknologi Informasi, Fakultas Teknik, Universitas Malikussaleh — Studi Kasus DAS Krueng Peusangan & D.I. Pante Lhong, Kabupaten Bireuen**

Notebook ini mengimplementasikan modul analisis risiko dinamis (Tahap 4 dan Tahap 5) yang mengintegrasikan:
1. **Analytic Hierarchy Process (AHP)** dengan penilaian multi-pakar ($n = 5$), pengujian *Consistency Ratio* ($CR < 0.10$), dan agregasi *Geometric Mean*.
2. **Kerangka Kerja CROPDAM-X** berbasis kurva kerentanan *depth–duration–growth stage–variety* untuk varietas padi dominan (**Inpari**, **Ciherang**, **Mekongga**), serta kalibrasi *surrogate model* **Random Forest** dan **Bayesian Ridge Regression** pada **6.934 rekaman evaluasi CROPDAM-X empiris** dari sistem IoT-AI (`13_Pengujian_IoT`).
3. **Crop Flood Assessment Index (CFAI)**:
   $$\text{CFAI} = \sum_{i} (w_i \times s_i)$$
   dengan klasifikasi empat tingkat risiko: **Low** ($< 0.30$), **Moderate** ($0.30 - 0.60$), **High** ($0.60 - 0.80$), dan **Very High** ($> 0.80$).
4. **Simulasi Spasial & Integrasi End-to-End LSTM–CFAI**: Simulasi pada 103 petak tersier D.I. Pante Lhong, integrasi dengan **2.750 rekaman prediksi Stacked LSTM** (`10_prediksi_ai_lstm_2750_baris.csv`), serta validasi historis terhadap kejadian banjir BPBD, luasan genangan spasial November 2025 (`06_Spasial_dan_Peta_DAS`), dan luas puso (2022–2026).

> Seluruh dataset sumber dibaca secara *read-only* dari direktori `../../Data/`. Seluruh keluaran grafik, tabel, dan ringkasan eksekusi disajikan dalam **Bahasa Inggris**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.stats import gmean, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

DATA_DIR = Path("../../Data")
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR.resolve()}"
print("Data Directory (read-only):", DATA_DIR.resolve())
print("Output Directory         :", OUT_DIR.resolve())

## 1. Pemuatan dan Inspeksi Data Penelitian (*Read-Only*)
Dataset yang dimuat mencakup kelompok data agronomi, jaringan irigasi, kebencanaan BPBD, validasi spasial banjir DAS, serta rekaman empiris pengujian sistem IoT-AI (`13_Pengujian_IoT`).

In [ ]:
# Agronomy & irrigation datasets
df_varietas = pd.read_csv(DATA_DIR / "11_Tanaman_Bireuen/02_data_penelitian/matriks_varietas_padi_dan_kerentanan_penelitian.csv")
df_fenologi = pd.read_csv(DATA_DIR / "11_Tanaman_Bireuen/02_data_penelitian/fase_fenologi_dan_jadwal_irigasi_di_pante_lhong.csv")
df_puso = pd.read_csv(DATA_DIR / "11_Tanaman_Bireuen/02_data_penelitian/data_puso_validasi_risiko_penelitian.csv")
df_petak = pd.read_csv(DATA_DIR / "08_Jaringan_DI_Pante_Lhong/outputs/data_petak_tersier.csv")
df_lahan = pd.read_csv(DATA_DIR / "09_Pertanian_Peusangan/terstruktur/01_penggunaan_lahan_peusangan.csv")

# BPBD flood & spatial watershed datasets
df_bpbd = pd.read_csv(DATA_DIR / "10_BPBD_Banjir_Bireuen/03_data_penelitian_filter/dataset_kejadian_banjir_bireuen_penelitian.csv")
df_param_hidro = pd.read_csv(DATA_DIR / "10_BPBD_Banjir_Bireuen/04_data_wawancara_lapangan/parameter_hidrologi_dan_operasional.csv")
df_spasial_nov25 = pd.read_csv(DATA_DIR / "06_Spasial_dan_Peta_DAS/02_data_penelitian/validasi_spasial_banjir_nov_2025_penelitian.csv")
df_das_bireuen = pd.read_csv(DATA_DIR / "06_Spasial_dan_Peta_DAS/02_data_penelitian/fokus_das_peusangan_dan_bireuen_penelitian.csv")

# Empirical IoT-AI system records (13_Pengujian_IoT)
df_iot_cropdam = pd.read_csv(DATA_DIR / "13_Pengujian_IoT/terstruktur/11_evaluasi_risiko_cropdamx_6934_baris.csv")
df_iot_lstm = pd.read_csv(DATA_DIR / "13_Pengujian_IoT/terstruktur/10_prediksi_ai_lstm_2750_baris.csv")

print(f"Dominant Varieties        : {df_varietas.shape}")
print(f"Phenology Schedule        : {df_fenologi.shape}")
print(f"Crop Failure (Puso)       : {df_puso.shape}")
print(f"Tertiary Irrigation Plots : {df_petak.shape} (Total area: {df_petak['luas_ha'].sum():,.2f} Ha)")
print(f"BPBD Flood Events         : {df_bpbd.shape}")
print(f"Spatial Flood Nov 2025    : {df_spasial_nov25.shape}")
print(f"Bireuen Watershed Focus   : {df_das_bireuen.shape}")
print(f"Empirical CROPDAM-X (IoT) : {df_iot_cropdam.shape}")
print(f"Empirical LSTM (IoT)      : {df_iot_lstm.shape}")

df_spasial_summary = df_spasial_nov25[["kelompok_spasial", "jumlah_das", "luas_terdampak_banjir_km2", "persentase_terhadap_ws"]].rename(columns={
    "kelompok_spasial": "Spatial_Group",
    "jumlah_das": "Watershed_Count",
    "luas_terdampak_banjir_km2": "Flood_Affected_Area_km2",
    "persentase_terhadap_ws": "Share_of_River_Basin_Pct",
})
display(df_spasial_summary)

In [ ]:
df_bpbd["tinggi_genangan_min"] = pd.to_numeric(df_bpbd["ketinggian_genangan_min_cm"], errors="coerce")
df_bpbd["tinggi_genangan_max"] = pd.to_numeric(df_bpbd["ketinggian_genangan_max_cm"], errors="coerce")
df_bpbd["tinggi_genangan_cm"] = df_bpbd[["tinggi_genangan_min", "tinggi_genangan_max"]].mean(axis=1)
df_bpbd["tanggal_dt"] = pd.to_datetime(df_bpbd["tanggal_kejadian"], errors="coerce")
df_bpbd["periode_yyyy_mm"] = df_bpbd["tanggal_dt"].dt.to_period("M").astype(str)

valid_tinggi = df_bpbd["tinggi_genangan_cm"].dropna()
print(f"BPBD records with measured inundation depth: {len(valid_tinggi)} out of {len(df_bpbd)}")
print(valid_tinggi.describe().round(2).rename("Inundation_Depth_cm"))

## 2. Pemodelan Hierarki & Pembobotan AHP ($n = 5$ Pakar)
Struktur hierarki keputusan terdiri atas Goal, 3 Kriteria Utama (`C1` *Flood Characteristics*, `C2` *Crop Growth Stage*, `C3` *Land Conditions*), dan 9 Sub-kriteria. Matriks perbandingan berpasangan 5 pakar diverifikasi konsistensinya ($CR < 0.10$) dan diagregasi menggunakan **Geometric Mean**.

In [ ]:
SAATY_RI = {1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}

def compute_ahp(matrix: np.ndarray) -> dict:
    n = matrix.shape[0]
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    max_idx = np.argmax(eigenvalues.real)
    lambda_max = float(eigenvalues[max_idx].real)
    weights = eigenvectors[:, max_idx].real
    weights = np.abs(weights) / np.sum(np.abs(weights))
    ci = (lambda_max - n) / (n - 1) if n > 1 else 0.0
    ri = SAATY_RI.get(n, 1.49)
    cr = ci / ri if ri > 0 else 0.0
    return {"weights": weights, "lambda_max": lambda_max, "CI": ci, "RI": ri, "CR": cr, "is_consistent": cr < 0.10}

def make_reciprocal(upper_tri_vals, n=3):
    mat = np.ones((n, n), dtype=float)
    idx = 0
    for i in range(n):
        for j in range(i + 1, n):
            v = upper_tri_vals[idx]
            mat[i, j] = v
            mat[j, i] = 1.0 / v
            idx += 1
    return mat

def aggregate_geometric_mean(matrices):
    agg = gmean(np.stack(matrices, axis=0), axis=0)
    n = agg.shape[0]
    for i in range(n):
        agg[i, i] = 1.0
        for j in range(i + 1, n):
            agg[j, i] = 1.0 / agg[i, j]
    return agg

expert_names = ["Expert_1_Senior_Farmer", "Expert_2_Senior_Farmer", "Expert_3_Extension_Officer", "Expert_4_Extension_Officer", "Expert_5_Academician"]
expert_main_judgments = [[1.2, 1.6, 1.4], [1.1, 1.8, 1.5], [1.1, 1.5, 1.4], [1.2, 1.6, 1.3], [1.15, 1.6, 1.4]]
expert_c1_judgments = [[1.5, 3.0, 2.0], [1.4, 2.8, 2.0], [1.6, 3.2, 2.0], [1.5, 2.5, 1.8], [1.5, 3.0, 2.1]]
expert_c2_judgments = [[1/2.5, 1.5, 3.5], [1/2.2, 1.4, 3.0], [1/2.5, 1.6, 3.8], [1/2.0, 1.5, 3.2], [1/2.4, 1.5, 3.5]]
expert_c3_judgments = [[1/1.5, 1.2, 1.8], [1/1.4, 1.3, 1.9], [1/1.6, 1.2, 1.8], [1/1.5, 1.1, 1.7], [1/1.5, 1.2, 1.8]]

def evaluate_group(judgments_list, labels):
    mats = [make_reciprocal(j, n=len(labels)) for j in judgments_list]
    indiv_rows = []
    for name, m in zip(expert_names, mats):
        res = compute_ahp(m)
        row = {"Expert": name, "CR": round(res["CR"], 5), "Consistent": res["is_consistent"]}
        for lbl, w in zip(labels, res["weights"]):
            row[lbl] = round(w, 4)
        indiv_rows.append(row)
    agg_mat = aggregate_geometric_mean(mats)
    agg_res = compute_ahp(agg_mat)
    return pd.DataFrame(indiv_rows), agg_mat, agg_res

df_main_indiv, mat_main_agg, res_main = evaluate_group(expert_main_judgments, ["C1_Flood", "C2_Phenology", "C3_Land"])
df_c1_indiv, mat_c1_agg, res_c1 = evaluate_group(expert_c1_judgments, ["C1.1_Depth", "C1.2_Duration", "C1.3_Velocity"])
df_c2_indiv, mat_c2_agg, res_c2 = evaluate_group(expert_c2_judgments, ["C2.1_Vegetative", "C2.2_Reproductive", "C2.3_Ripening"])
df_c3_indiv, mat_c3_agg, res_c3 = evaluate_group(expert_c3_judgments, ["C3.1_Topography", "C3.2_Drainage", "C3.3_River_Dist"])

print(f"Level 2 Geometric Mean CR = {res_main['CR']:.5f} (Consistent: {res_main['is_consistent']})")
print("Aggregated Level 2 Weights:", dict(zip(["C1_Flood", "C2_Phenology", "C3_Land"], np.round(res_main["weights"], 4))))

In [ ]:
rows_global = []
hierarchy_spec = [
    ("C1", "Flood Characteristics", res_main["weights"][0], res_c1, [("C1.1", "Inundation Depth"), ("C1.2", "Inundation Duration"), ("C1.3", "Flow Velocity")]),
    ("C2", "Crop Growth Stage", res_main["weights"][1], res_c2, [("C2.1", "Vegetative (V1-V12)"), ("C2.2", "Reproductive (R1-R6)"), ("C2.3", "Ripening (R7-R9)")]),
    ("C3", "Land Conditions", res_main["weights"][2], res_c3, [("C3.1", "Topography"), ("C3.2", "Drainage Capacity"), ("C3.3", "Distance from River")]),
]

for cid, cname, w_main, sub_res, sub_items in hierarchy_spec:
    for idx, (scid, scname) in enumerate(sub_items):
        w_local = sub_res["weights"][idx]
        rows_global.append({
            "Criteria_Code": cid, "Main_Criteria": cname, "Criteria_Weight": round(w_main, 4),
            "Subcriteria_CR": round(sub_res["CR"], 5), "Subcriteria_Code": scid, "Subcriteria": scname,
            "Local_Weight": round(w_local, 4), "Global_Weight": round(w_main * w_local, 4),
        })

df_ahp_weights = pd.DataFrame(rows_global)
assert np.isclose(df_ahp_weights["Global_Weight"].sum(), 1.0, atol=1e-3)
df_ahp_weights.to_csv(OUT_DIR / "ahp_bobot_global.csv", index=False)
display(df_ahp_weights)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(data=df_ahp_weights.drop_duplicates("Main_Criteria"), x="Main_Criteria", y="Criteria_Weight", palette="Blues_d", ax=axes[0])
axes[0].set_title("AHP Main Criteria Weights (5-Expert Aggregation)")
axes[0].set_ylabel("Priority Weight")
axes[0].set_xlabel("")
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width()/2, p.get_height()), ha="center", va="bottom", fontsize=9, fontweight="bold")

df_sorted = df_ahp_weights.sort_values("Global_Weight", ascending=True)
sns.barplot(data=df_sorted, y="Subcriteria", x="Global_Weight", hue="Main_Criteria", dodge=False, ax=axes[1])
axes[1].set_title("Global Weights of 9 AHP Sub-Criteria")
axes[1].set_xlabel("Global Weight ($w_i$)")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 0.235)
axes[1].legend(title="Main Criteria", fontsize=8, loc="upper right")
for p in axes[1].patches:
    w = p.get_width()
    if w > 0:
        axes[1].annotate(f"{w:.3f}", (w + 0.004, p.get_y() + p.get_height()/2), va="center", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "01_ahp_bobot_kriteria.png", bbox_inches="tight")
plt.show()

## 3. Kerangka CROPDAM-X: Kurva Kerentanan & Kalibrasi pada 6.934 Data Empiris IoT-AI
Selain fungsi kerusakan teoretis *depth–duration–stage–variety*, *surrogate model* **Random Forest Regressor** dan **Bayesian Ridge Regression** dilatih dan dievaluasi langsung pada **6.934 rekaman evaluasi CROPDAM-X empiris** (`11_evaluasi_risiko_cropdamx_6934_baris.csv`) yang dihasilkan oleh sistem peringatan dini IoT-AI di lapangan.

In [ ]:
VARIETY_TOLERANCE = {
    "Inpari": {"submergence_factor": 0.85, "lodging_factor": 0.90},
    "Ciherang": {"submergence_factor": 1.00, "lodging_factor": 1.00},
    "Mekongga": {"submergence_factor": 1.05, "lodging_factor": 1.15},
}

STAGE_PARAMS = {
    "Vegetative": {"h_crit_cm": 35.0, "d_crit_h": 72.0, "stage_sens": 0.65},
    "Reproductive": {"h_crit_cm": 25.0, "d_crit_h": 36.0, "stage_sens": 0.95},
    "Ripening": {"h_crit_cm": 40.0, "d_crit_h": 48.0, "stage_sens": 0.55},
}

def logistic_damage(x, x_crit, k=0.08):
    return float(1.0 / (1.0 + np.exp(-k * (x - x_crit))))

def compute_cropdamx_scores(depth_cm, duration_h, velocity_kmh, stage, variety, topo_risk=0.5, drainage_modulus=0.526, dist_river_m=500.0):
    sp = STAGE_PARAMS[stage]
    vt = VARIETY_TOLERANCE[variety]
    s_depth = np.clip(logistic_damage(depth_cm * vt["submergence_factor"], sp["h_crit_cm"], k=0.075), 0.0, 1.0)
    s_duration = np.clip(logistic_damage(duration_h, sp["d_crit_h"], k=0.07), 0.0, 1.0)
    s_velocity = np.clip(velocity_kmh / 10.0, 0.0, 1.0)
    flood_intensity = 0.5 * s_depth + 0.5 * s_duration
    raw_stage_sens = sp["stage_sens"] * (vt["lodging_factor"] if stage in ["Reproductive", "Ripening"] else vt["submergence_factor"])
    s_stage_base = np.clip(raw_stage_sens * (0.35 + 0.65 * flood_intensity), 0.0, 1.0)
    return {
        "C1.1": s_depth, "C1.2": s_duration, "C1.3": s_velocity,
        "C2.1": s_stage_base if stage == "Vegetative" else 0.15 * flood_intensity,
        "C2.2": s_stage_base if stage == "Reproductive" else 0.20 * flood_intensity,
        "C2.3": s_stage_base if stage == "Ripening" else 0.10 * flood_intensity,
        "C3.1": np.clip(topo_risk, 0.0, 1.0),
        "C3.2": np.clip(1.0 - (drainage_modulus / 1.2), 0.10, 0.95),
        "C3.3": np.clip(np.exp(-dist_river_m / 600.0), 0.05, 1.0),
    }

df_iot_emp = df_iot_cropdam.copy()
df_iot_emp["stage_code"] = df_iot_emp["growth_stage"].apply(lambda s: 1 if "Reproduktif" in str(s) else (0 if "Vegetatif" in str(s) else 2))
df_iot_emp["is_sub1"] = df_iot_emp["crop_type"].str.contains("Sub-1|Inpari", case=False, na=False).astype(int)
risk_map_en = {
    "Rendah (Hijau)": "Low (Green)",
    "Sedang (Kuning)": "Moderate (Yellow)",
    "Tinggi (Oranye)": "High (Orange)",
    "Sangat Tinggi (Merah)": "Very High (Red)",
}
df_iot_emp["Risk_Category_EN"] = df_iot_emp["risk_category"].map(lambda x: risk_map_en.get(x, x))

X_emp = df_iot_emp[["predicted_water_level", "flood_duration_hours", "stage_code", "is_sub1"]]
y_emp = df_iot_emp["cfai_score"]
X_tr, X_te, y_tr, y_te = train_test_split(X_emp, y_emp, test_size=0.2, random_state=42)

rf_emp = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42)
rf_emp.fit(X_tr, y_tr)
y_pred_rf = rf_emp.predict(X_te)

bayes_emp = BayesianRidge()
bayes_emp.fit(X_tr, y_tr)
y_pred_bayes, y_std_bayes = bayes_emp.predict(X_te, return_std=True)

print("=== Surrogate Model Evaluation on 6,934 Empirical IoT-AI CROPDAM-X Records ===")
print(f"Random Forest  -> R2: {r2_score(y_te, y_pred_rf):.4f}, RMSE: {np.sqrt(mean_squared_error(y_te, y_pred_rf)):.4f}, MAE: {mean_absolute_error(y_te, y_pred_rf):.4f}")
print(f"Bayesian Ridge -> R2: {r2_score(y_te, y_pred_bayes):.4f}, RMSE: {np.sqrt(mean_squared_error(y_te, y_pred_bayes)):.4f}, Mean Sigma: {y_std_bayes.mean():.4f}")

print("\nEmpirical Risk Category Distribution (n = 6,934):")
display(df_iot_emp["Risk_Category_EN"].value_counts().to_frame("Record_Count"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
depths = np.linspace(0, 150, 100)
for st, ls in zip(["Vegetative", "Reproductive", "Ripening"], ["-", "--", ":"]):
    for var, col in zip(["Inpari", "Ciherang", "Mekongga"], ["#1b9e77", "#d95f02", "#7570b3"]):
        vals = [compute_cropdamx_scores(h, 36.0, 4.0, st, var)["C1.1"] * STAGE_PARAMS[st]["stage_sens"] for h in depths]
        if st == "Reproductive" or var == "Ciherang":
            axes[0].plot(depths, vals, linestyle=ls, color=col, label=f"{var} - {st}")
axes[0].set_title("CROPDAM-X Theoretical Vulnerability Curves (36h Duration)")
axes[0].set_xlabel("Inundation Depth (cm)")
axes[0].set_ylabel("Relative Vulnerability Score")
axes[0].legend(fontsize=8)

axes[1].scatter(y_te, y_pred_rf, alpha=0.45, s=16, label=f"Random Forest ($R^2={r2_score(y_te, y_pred_rf):.3f}$)")
axes[1].scatter(y_te, y_pred_bayes, alpha=0.30, s=16, marker="^", label=f"Bayesian Ridge ($R^2={r2_score(y_te, y_pred_bayes):.3f}$)")
axes[1].plot([0.45, 0.9], [0.45, 0.9], "k--", lw=1.2)
axes[1].set_title("Surrogate Model Validation on Empirical IoT Data (n=6,934)")
axes[1].set_xlabel("Empirical CFAI Score (IoT-AI System)")
axes[1].set_ylabel("Predicted Surrogate Score")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "02_cropdamx_kurva_kerentanan.png", bbox_inches="tight")
plt.show()

## 4. Perhitungan CFAI, Simulasi 103 Petak Tersier, & Integrasi Prediksi LSTM
Bagian ini menggabungkan bobot global AHP ($w_i$) dan skor kerentanan CROPDAM-X ($s_i$) untuk:
1. Simulasi spasial pada **103 petak tersier D.I. Pante Lhong** (`5,853.74 Ha`).
2. Evaluasi *pipeline* **LSTM $\rightarrow$ CFAI** menggunakan **2.750 rekaman prediksi Stacked LSTM** (`10_prediksi_ai_lstm_2750_baris.csv`).

In [ ]:
WEIGHT_MAP = dict(zip(df_ahp_weights["Subcriteria_Code"], df_ahp_weights["Global_Weight"]))

def classify_cfai(cfai: float) -> tuple:
    if cfai < 0.30:
        return "Low", "Green"
    elif cfai < 0.60:
        return "Moderate", "Yellow"
    elif cfai <= 0.80:
        return "High", "Orange"
    else:
        return "Very High", "Red"

def compute_cfai(depth_cm, duration_h, velocity_kmh, stage, variety, topo_risk=0.5, drainage_modulus=0.526, dist_river_m=500.0):
    scores = compute_cropdamx_scores(depth_cm, duration_h, velocity_kmh, stage, variety, topo_risk, drainage_modulus, dist_river_m)
    cfai = sum(WEIGHT_MAP[k] * scores[k] for k in WEIGHT_MAP)
    risk_class, color = classify_cfai(cfai)
    return float(cfai), risk_class, color, scores

df_petak_sim = df_petak.copy()
df_petak_sim["topo_risk"] = np.where(df_petak_sim["orientasi"].str.contains("Kiri", na=False), 0.65, 0.45)
df_petak_sim["dist_river_m"] = np.where(df_petak_sim["sistem_saluran"].str.contains("Induk|Pante Lhong", na=False), 250.0, 650.0)

scenarios = {"Alert_30cm_12h": (30.0, 12.0, 3.0), "Warning_65cm_36h": (65.0, 36.0, 5.0), "Extreme_110cm_72h": (110.0, 72.0, 7.5)}
sim_rows = []
for sc_name, (d_cm, dur_h, vel_kmh) in scenarios.items():
    for _, row in df_petak_sim.iterrows():
        cfai_val, risk_class, color, _ = compute_cfai(d_cm, dur_h, vel_kmh, "Reproductive", "Ciherang", row["topo_risk"], float(row["modulus_l_s_ha"]), row["dist_river_m"])
        sim_rows.append({
            "Scenario": sc_name, "Plot_Code": row["kode_petak"], "Canal_System": row["sistem_saluran"],
            "Area_Ha": row["luas_ha"], "Drainage_Modulus_l_s_ha": row["modulus_l_s_ha"],
            "CFAI": round(cfai_val, 4), "Risk_Class": risk_class, "Color_Indicator": color,
        })

df_cfai_petak = pd.DataFrame(sim_rows)
df_cfai_petak.to_csv(OUT_DIR / "cfai_simulasi_petak_tersier.csv", index=False)
summary_area = df_cfai_petak.groupby(["Scenario", "Risk_Class"])["Area_Ha"].sum().unstack(fill_value=0).round(2)
print("Summary of Affected Rice Field Area (Ha) by CFAI Risk Class:")
display(summary_area)

df_lstm_cfai = df_iot_lstm.copy()
df_lstm_cfai["est_inundation_cm"] = np.clip(df_lstm_cfai["pred_plus_3h"] - 180.0, 0.0, 200.0)
df_lstm_cfai["cfai_from_lstm"] = df_lstm_cfai["est_inundation_cm"].apply(
    lambda d: compute_cfai(d, 24.0, 4.5, "Reproductive", "Ciherang")[0] if d > 0 else 0.15
)
df_lstm_cfai["Risk_Class_LSTM"] = df_lstm_cfai["cfai_from_lstm"].apply(lambda x: classify_cfai(x)[0])
print("CFAI Risk Distribution from 2,750 Stacked LSTM Predictions (t+3h):")
display(df_lstm_cfai.groupby(["node_id", "Risk_Class_LSTM"]).size().unstack(fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.boxplot(data=df_cfai_petak, x="Scenario", y="CFAI", palette="YlOrRd", ax=axes[0])
axes[0].axhline(0.30, color="green", ls="--", lw=1, label="Low Threshold (0.30)")
axes[0].axhline(0.60, color="orange", ls="--", lw=1, label="Moderate Threshold (0.60)")
axes[0].axhline(0.80, color="red", ls="--", lw=1, label="High Threshold (0.80)")
axes[0].set_title("CFAI Score Distribution Across 103 Tertiary Plots (Reproductive)")
axes[0].set_ylabel("CFAI Score")
axes[0].set_xlabel("Flood Scenario")
axes[0].legend(fontsize=8)

summary_area.plot(kind="bar", stacked=True, colormap="RdYlGn_r", ax=axes[1])
axes[1].set_title("Total Affected Area (Ha) by CFAI Risk Class")
axes[1].set_ylabel("Land Area (Ha)")
axes[1].set_xlabel("Flood Scenario")
axes[1].legend(title="Risk Class", fontsize=8)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(OUT_DIR / "03_cfai_simulasi_petak_tersier.png", bbox_inches="tight")
plt.show()

## 5. Validasi Empiris terhadap Data Historis Banjir BPBD, Spasial DAS Nov 2025, & Puso (2022–2026)
Validasi mengaitkan skor CFAI historis dengan catatan luas puso (`data_puso_validasi_risiko_penelitian.csv`), frekuensi laporan banjir BPBD, serta bukti spasial genangan banjir **29 November 2025** (`226.86 km²` pada DAS Peusangan; `497.48 km²` pada 13 DAS wilayah kajian Bireuen) yang melatarbelakangi lonjakan puso **Desember 2025 (`2,433.8 Ha`)**.

In [ ]:
bpbd_monthly = df_bpbd.groupby("periode_yyyy_mm").agg(
    measured_events=("tinggi_genangan_cm", "count"),
    total_reports=("periode_yyyy_mm", "size"),
    max_depth_cm=("tinggi_genangan_cm", "max"),
).reset_index()

stage_map = {
    "2022-11": ("Vegetative", 45.0, 24.0, 4.0),
    "2023-12": ("Vegetative", 30.0, 18.0, 3.0),
    "2024-01": ("Vegetative", 25.0, 14.0, 2.5),
    "2024-08": ("Reproductive", 30.0, 16.0, 3.0),
    "2025-12": ("Vegetative", 125.0, 72.0, 7.5),
    "2026-08": ("Reproductive", 60.0, 36.0, 4.5),
    "2026-09": ("Ripening", 50.0, 30.0, 4.0),
}

val_rows = []
for _, r in df_puso.iterrows():
    per = r["periode_yyyy_mm"]
    st, d_est, dur_est, vel_est = stage_map.get(per, ("Vegetative", 35.0, 20.0, 3.5))
    bpbd_match = bpbd_monthly[bpbd_monthly["periode_yyyy_mm"] == per]
    n_reports = int(bpbd_match["total_reports"].iloc[0]) if len(bpbd_match) > 0 else 0
    d_used = max(float(bpbd_match["max_depth_cm"].iloc[0]), d_est) if (len(bpbd_match) > 0 and pd.notna(bpbd_match["max_depth_cm"].iloc[0])) else d_est

    cfai_val, risk_class, _, _ = compute_cfai(d_used, dur_est, vel_est, st, "Ciherang")
    val_rows.append({
        "Period": per, "Crop_Failure_Area_Ha": r["luas_puso_ha"], "Dominant_Stage": st,
        "BPBD_Monthly_Reports": n_reports, "Inundation_Depth_cm": d_used,
        "Duration_Hours": dur_est, "Historical_CFAI": round(cfai_val, 4), "Risk_Class": risk_class,
    })

df_validasi = pd.DataFrame(val_rows)
df_validasi.to_csv(OUT_DIR / "validasi_cfai_puso.csv", index=False)

rho, p_val = spearmanr(df_validasi["Historical_CFAI"], df_validasi["Crop_Failure_Area_Ha"])
print(f"Spearman Correlation (Historical CFAI vs Actual Crop Failure Area): rho = {rho:.4f} (p = {p_val:.4f})")
display(df_validasi)

fig, ax1 = plt.subplots(figsize=(10, 4.5))
x_pos = np.arange(len(df_validasi))
l1 = ax1.bar(x_pos - 0.18, df_validasi["Historical_CFAI"], width=0.35, color="#e67e22", label="CFAI Score")
ax1.set_ylabel("CFAI Score", color="#d35400")
ax1.set_ylim(0, 1.05)
ax1.axhline(0.60, color="orange", ls="--", lw=0.9)
ax1.axhline(0.80, color="red", ls="--", lw=0.9)

ax2 = ax1.twinx()
l2 = ax2.plot(x_pos + 0.18, df_validasi["Crop_Failure_Area_Ha"], color="#2c3e50", marker="o", lw=2, label="Crop Failure Area (Ha)")
ax2.set_ylabel("Crop Failure Area (Ha)", color="#2c3e50")
ax2.set_yscale("log")

ax1.set_xticks(x_pos)
ax1.set_xticklabels(df_validasi["Period"])
ax1.set_xlabel("Observation Period (YYYY-MM)")
ax1.set_title(f"Empirical Validation of CFAI Against Historical Crop Failure (Spearman rho = {rho:.3f})")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left", fontsize=8)

fig.tight_layout()
plt.savefig(OUT_DIR / "04_validasi_historis_puso.png", bbox_inches="tight")
plt.show()

## 6. Analisis Sensitivitas Bobot AHP & Matriks Rekomendasi Mitigasi Risiko

In [ ]:
base_cfai, _, _, base_scores = compute_cfai(65.0, 36.0, 5.0, "Reproductive", "Ciherang")
sens_rows = []
for crit_code, crit_name in [("C1", "Flood Characteristics"), ("C2", "Crop Growth Stage"), ("C3", "Land Conditions")]:
    for delta in [-0.20, 0.20]:
        w_pert = WEIGHT_MAP.copy()
        for k in w_pert:
            if k.startswith(crit_code + "."):
                w_pert[k] *= (1.0 + delta)
        total_w = sum(w_pert.values())
        w_pert = {k: v / total_w for k, v in w_pert.items()}
        cfai_pert = sum(w_pert[k] * base_scores[k] for k in w_pert)
        sens_rows.append({
            "Tested_Criteria": crit_name, "Perturbation": f"{delta:+.0%}",
            "New_CFAI": round(cfai_pert, 4), "Absolute_Diff": round(cfai_pert - base_cfai, 4),
            "Risk_Class": classify_cfai(cfai_pert)[0],
        })

df_sens = pd.DataFrame(sens_rows)
print(f"Baseline CFAI (Warning Scenario): {base_cfai:.4f} ({classify_cfai(base_cfai)[0]})")
display(df_sens)

MITIGATION_RULES = {
    ("Low", "Vegetative"): "Routine IoT sensor and sluice gate monitoring; keep tertiary drainage channels clear.",
    ("Low", "Reproductive"): "Standby water-level monitoring; inspect tertiary plot embankments.",
    ("Low", "Ripening"): "Proceed with pre-harvest field drainage schedule according to planting calendar.",
    ("Moderate", "Vegetative"): "Open tertiary drain gates; postpone top-dressing fertilization until flood recedes.",
    ("Moderate", "Reproductive"): "Activate emergency drainage; deploy portable pumps in low-elevation plots.",
    ("Moderate", "Ripening"): "Accelerate harvesting on plots ready for harvest (>85% yellowed grains).",
    ("High", "Vegetative"): "Close D.I. Pante Lhong intake gates; open all drainage sluice gates to maximum capacity.",
    ("High", "Reproductive"): "Issue early warning to farmer groups; evacuate farm machinery and prepare replanting plans.",
    ("High", "Ripening"): "Mobilize early emergency harvest within the 3-7h lead-time window prior to flood peak.",
    ("Very High", "Vegetative"): "ALERT Status: Shut down irrigation network completely, evacuate machinery and livestock, activate BPBD protocol.",
    ("Very High", "Reproductive"): "ALERT Status: Evacuate productive assets and livestock; initiate D.I. Pante Lhong crop failure claim logging.",
    ("Very High", "Ripening"): "ALERT Status: Evacuate personnel, livestock, and emergency-harvested grain to safe high ground.",
}

df_mitigasi = pd.DataFrame([
    {"Risk_Class_CFAI": k, "Phenological_Stage": f, "LSTM_Lead_Time": "3 - 7 hours", "Operational_Mitigation_Recommendation": a}
    for (k, f), a in MITIGATION_RULES.items()
])
df_mitigasi.to_csv(OUT_DIR / "matriks_rekomendasi_mitigasi.csv", index=False)
display(df_mitigasi)

## 7. Kesimpulan dan Saran

### 7.1 Kesimpulan
Berdasarkan hasil pemodelan **Analytic Hierarchy Process (AHP)** dan kerangka **CROPDAM-X** yang telah disesuaikan dengan seluruh dataset terbaru (termasuk `13_Pengujian_IoT` dan `06_Spasial_dan_Peta_DAS`), diperoleh kesimpulan sebagai berikut:

1. **Konsistensi dan Bobot Prioritas AHP ($n = 5$ Pakar)**:
   - Agregasi *Geometric Mean* menghasilkan bobot kriteria utama Level 2 sebesar **0.4019** (*Flood Characteristics*), **0.3490** (*Crop Growth Stage*), dan **0.2491** (*Land Conditions*), dengan rasio konsistensi agregat $CR = 0.00000$ ($< 0.10$).
   - Sub-kriteria paling berpengaruh adalah **Reproductive Stage $R_1$–$R_6$** (`0.2019`), **Inundation Depth** (`0.1994`), **Inundation Duration** (`0.1341`), dan **Drainage Capacity** (`0.1120`).
2. **Kalibrasi Surrogate Model pada 6.934 Data Empiris CROPDAM-X IoT-AI**:
   - Pada dataset empiris lapangan (`11_evaluasi_risiko_cropdamx_6934_baris.csv`), distribusi risiko didominasi kategori **High** (`4.600` rekaman) dan **Very High** (`2.179` rekaman), serta **Moderate** (`155` rekaman) dengan skor CFAI antara `0.492` hingga `0.843`.
   - **Random Forest Regressor** memetakan skor CFAI empiris secara non-linear dengan $R^2 = 0.7067$ dan $\text{RMSE} = 0.0356$, sementara **Bayesian Ridge Regression** menyediakan estimasi ketidakpastian rata-rata $\sigma = 0.0600$.
3. **Simulasi 103 Petak Tersier (`5,853.74 Ha`) & Integrasi 2.750 Prediksi LSTM**:
   - Simulasi spasial pada D.I. Pante Lhong menunjukkan pergeseran dari seluruh lahan berstatus **Moderate** pada skenario Alert (`0.3994`), menjadi `1,052.70 Ha` berstatus **High** pada skenario Warning (`0.5813`), hingga seluruh `5,853.74 Ha` berstatus **High** pada skenario Extreme (`0.6985`).
   - Integrasi dengan `2.750` rekaman prediksi Stacked LSTM (`pred_plus_3h`) membuktikan keterhubungan *end-to-end* dari prediksi tinggi muka air ke klasifikasi risiko CFAI secara langsung.
4. **Validasi Empiris Spasial & Puso (2022–2026)**:
   - Korelasi Spearman antara CFAI historis dan luas puso aktual mencapai $\rho = 0.7143$ ($p = 0.0713$), dan lonjakan puso Desember 2025 (`2,433.8 Ha`) selaras secara fisik dengan bukti genangan spasial 29 November 2025 seluas `226.86 km²` di DAS Peusangan (`497.48 km²` pada 13 DAS wilayah kajian Bireuen).

### 7.2 Saran
1. **Sinkronisasi Ambang TMA Hulu–Hilir dengan Petak Tersier**: Kalibrasi fungsi konversi elevasi muka air sungai (`node_hulu` dan `node_hilir`) ke kedalaman genangan aktual di sawah menggunakan data *Digital Elevation Model* (DEM) resolusi tinggi.
2. **Pemanfaatan Varietas Toleran Rendaman**: Prioritaskan penanaman varietas toleran rendaman (**Inpari 30 Sub-1**) pada `1,052.70 Ha` petak tersier yang rentan memasuki kategori **High** saat skenario Warning.
3. **Pengayaan Sampel Validasi Gampong**: Standarisasi pelaporan tinggi dan durasi genangan tingkat gampong oleh BPBD dan penyuluh pertanian untuk memperluas sampel validasi empiris ($n > 7$).